In [2]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings"


In [3]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

df_review = pd.DataFrame(rows).sort_values(["variable", "firm_id"]).reset_index(drop=True)

print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
display(df_review)

Manual review needed for 48 firm-variable mappings across 34 firms.


,firm_id,variable,notes,final_choice,num_candidates
0,AFK.OL,BE,For BE we must exclude non-controlling interes...,Balance Sheet::Capital and reserves attributab...,9
1,BEWI.OL,BE,Book equity should be mapped from the Balance ...,,2
2,GSFG.OL,BE,The most appropriate BE measure is equity attr...,Balance Sheet::Total controlling interests,6
3,CAPSL.OL,COGS,The provided candidate list does not include a...,Income Statement::Cost of contract fulfillment,5
4,CONTX.OL,COGS,The true direct cost of sales is best represen...,Income Statement::Goods for resale,5
5,GOD.OL,COGS,The provided COGS candidates are mostly SG&A o...,Income Statement::External projects costs,11
6,NAS.OL,COGS,COGS for an airline corresponds to direct oper...,Income Statement::Technical Maintenance Expens...,17
7,VAR.OL,COGS,The true COGS-equivalent line is 'Production c...,Income Statement::Production costs,10
8,VEND.OL,COGS,Best semantic match for COGS is 'Costs of good...,Income Statement::Costs of goods and services ...,3
9,ZAP.OL,COGS,The true cost-of-sales line appears to be 'Cos...,Income Statement::Cost of inventories,5
